In [2]:
"""
Step 1: importar planilha
Step 2: definir regras colunas
Step 3: verificar colunas
Step 4: marcar células afetadas
Step 5: exportar planilha p/ correção
"""

import pandas as pd
from openpyxl import load_workbook, Workbook
from openpyxl.styles import PatternFill

In [3]:
#Step 1: Importar Planilha
worksheet = pd.read_excel(r"C:\Users\User\Desktop\Controle Clientes.xlsx")
worksheet.dropna(axis = 0, how = "any", subset = "NOME", inplace = True)
worksheet.reset_index(drop = True, inplace = True)

In [4]:
#Step 2: definir regras colunas
#Step 3: verificar colunas
#Step 4: marcar células afetadas

#sets
valid_UF = {"AC", "AL", "AP", "AM", "BA", "CE", "ES", "DF", "GO", "MA",
                    "MG", "MS", "MT", "PA", "PB", "PE", "PR", "PI", "RJ", "RN",
                    "RS", "RO", "RR", "SC", "SP", "SE", "TO"}
no_caps = {"da", "de", "do", "das", "dos"}

#lists
error_UF = []
error_CIDADE = []
error_TELEFONE = []
error_TELEFONE2 =[]
error_COMPRADOR = []
error_ATENDENTE = []
error_EMAIL = []

for index, each in worksheet["UF"].items():
    if each not in valid_UF and pd.notna(each): #compara se UF planilha está no set válido
        error_UF.append(index) #anexa os inválidos à lista p/ correção futura
#print(f"UF: {error_UF}")

for index, each in worksheet["CIDADE"].items():
    if pd.notna(each) and (not isinstance(each, str) or
        not all(x.isalpha() or x.isspace() for x in each)):
        error_CIDADE.append(index)
        #A primeira linha do IF verifica se é Null ou NÃO É string, caso verdadeiro
        #marca como erro
        #A segunda linha verifica se É string e se seus valores são todos letras ou espaços
        #A ordem das verificações é importante para evitar erros:
        #Se for um float ou date, onde .isalpha não é válido, o código já valida o IF antes
        #de dar erro
    print(f"{worksheet.loc[index, "CIDADE"]}, {index}")


#print(f"cidade: {error_CIDADE}")

for index, each in worksheet["TELEFONE"].items():
    if pd.notna(each) and (type(each) != int or len(str(each)) not in {10, 11}):
        error_TELEFONE.append(index)
        #confirma formato e tamanho (ddd+fixo e ddd+celular)

#print(f"TEL: {error_TELEFONE}")

for index, each in worksheet["TELEFONE2"].items():
    if pd.notna(each) and (type(each) != int or len(str(each)) not in {10, 11}):
        error_TELEFONE2.append(index)
        #confirma formato e tamanho (ddd+fixo e ddd+celular)

#print(f"TEL2: {error_TELEFONE2}")

for index, each in worksheet["COMPRADOR"].items():
    if pd.notna(each) and (not isinstance(each, str) or
        not all(x.isalpha() or x.isspace() for x in each)):
        error_COMPRADOR.append(index)

#print(f"COMPRADOR: {error_COMPRADOR}")

for index, each in worksheet["ATENDENTE"].items():
    if pd.notna(each) and (not isinstance(each, str) or
        not all(x.isalpha() or x.isspace() for x in each)):
        error_ATENDENTE.append(index)

#print(f"ATT: {error_ATENDENTE}")

for index, each in worksheet["EMAIL"].items():
    if pd.notna(each) and (each.count("@") != 1
                       or each.count(".") < 1
                        or " " in each):
        error_EMAIL.append(index)

#print(f"EMAIL: {error_EMAIL}")

2026-09-02 00:00:00, 0
??, 1
??, 2
???, 3
???, 4
???, 5
???, 6
???, 7
???, 8
???, 9
???, 10
???, 11
???, 12
???, 13
Alumnínio, 14
Americana, 15
Americana, 16
Americana, 17
Americana, 18
Americana, 19
Americana, 20
Americana, 21
Americana, 22
Americana, 23
Americana, 24
Americana, 25
Americana, 26
Americana, 27
Americana, 28
Americana, 29
Americana, 30
Americana, 31
Americana, 32
Americana, 33
Americana, 34
Americana, 35
Americana, 36
Americana, 37
Americana, 38
Americana, 39
Americana, 40
Americana, 41
Americana, 42
Americana, 43
Americana, 44
Americana, 45
Americana, 46
Americana, 47
Americana, 48
Americana, 49
Americana, 50
Americana, 51
Americana, 52
Americana, 53
Americana, 54
Americana, 55
Americana, 56
Americana, 57
Anápolis, 58
Anápolis, 59
Anápolis, 60
Anchieta, 61
Aparecida de Goiania, 62
Aparecida de Goiania, 63
Aparecida de Goiania, 64
Aparecida de Goiania, 65
Aparecida de Goiania, 66
Aparecida de Goiania, 67
Aparecida de Goiania, 68
Aparecida de Goiania, 69
Apucarana, 70
Ar

In [5]:
#Step 5: criar, formatar, exportar planilha p/ correção

#definitions
wb = Workbook()
ws = wb.active

error_color = PatternFill(
    start_color="ffff99",
    end_color="ffff99",
    fill_type="solid"
)

#Fill WB
for col_num, col_name in enumerate(worksheet.columns, 1):
    ws.cell(row = 1, column = col_num, value = col_name)

for row_num, row in enumerate(worksheet.itertuples(index=False),2):
    for col_num, value in enumerate(row, 1):
        ws.cell(row = row_num, column = col_num, value = value)

#reseting colors
for row in ws.iter_rows():
    for cell in row:
        cell.fill = PatternFill(fill_type=None)

#Painting and formatting

for cell in ws["A"][1:]: #UF
    if cell.row in [index+2 for index in error_UF]:
        cell.fill = error_color
    #else:

for cell in ws["B"][1:]: #CIDADE
    if cell.row in [index+2 for index in error_CIDADE]:
        cell.fill = error_color
    elif isinstance(cell.value, float):
        print(cell.value)
    else:
        cell.value = " ".join(
            word.capitalize() if word.lower() not in no_caps
            else word.lower()
            for word in cell.value.split()
            )
        #Aproveita o loop em andamento para formatar a cidade
        #cria um split em cada palavra no campo
        #se for minúscula e não for no_caps, capitaliza
        #após verificar, faz um join

for cell in ws["C"][1:]: #NOME
    if isinstance(cell.value, str):
         cell.value = " ".join(
            word if word.isupper()
            else word.capitalize()
            for word in cell.value.split()
        )
for cell in ws["E"][1:]: #TELEFONE
    if cell.row in [index+2 for index in error_TELEFONE]:
        cell.fill = error_color
    elif isinstance(cell.value, int):
        each = str(cell.value)
        cell.value = f"({each[:2]}) {each[2:-4]}-{each[-4:]}"

for cell in ws["F"][1:]: #TELEFONE2
    if cell.row in [index+2 for index in error_TELEFONE2]:
        cell.fill = error_color
    elif isinstance(cell.value, int):
        each = str(cell.value)
        cell.value = f"({each[:2]}) {each[2:-4]}-{each[-4:]}"

for cell in ws["G"][1:]: #COMPRADOR
    if cell.row in [index+2 for index in error_COMPRADOR]:
        cell.fill = error_color
    elif isinstance(cell.value, float):
        print(cell.value)
    else:
        cell.value = " ".join(
            word if word.isupper()
            else word.capitalize()
            for word in cell.value.split()
        )
for cell in ws["I"][1:]: #ATENDENTE
    if cell.row in [index+2 for index in error_ATENDENTE]:
        cell.fill = error_color
    elif isinstance(cell.value, float):
        print(cell.value)
    else:
        cell.value = " ".join(
            word if word.isupper()
            else word.capitalize()
            for word in cell.value.split()
        )

for cell in ws["J"][1:]: #EMAIL
    if cell.row in [index+2 for index in error_EMAIL]:
        cell.fill = error_color

wb.save(r"C:\Users\User\Desktop\Atualização Controle Clientes-alt.xlsx")


nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
